In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import (
    DataLoader, Dataset, ConcatDataset, WeightedRandomSampler
)
from torchvision import datasets, transforms, models
import pandas as pd
import numpy as np
from PIL import Image

In [3]:
# Paths
DATA_DIR      = "/kaggle/input/competitions/3-lc-x-mnnit-ai-hackathon/Starter_Kit/data"
TRAIN_DIR     = DATA_DIR + "/train"
VAL_DIR       = DATA_DIR + "/val"
TEST_DIR      = DATA_DIR + "/test"
UNDEFINED_DIR = TRAIN_DIR + "/undefined"

In [4]:
IMAGE_SIZE = 128
BATCH_SIZE = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROUND_THRESHOLDS = [0.99, 0.97, 0.95]
print(f"Device : {DEVICE}")

Device : cpu


In [13]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.15),
    transforms.RandomGrayscale(p=0.15),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2)),  # ← AFTER ToTensor now
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

#TTA for maximum accuracy
tta_transforms = [
    # Normal
    val_transform,
    # Horizontal flip
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Vertical flip
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Both flips
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0), transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Center crop
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.15), int(IMAGE_SIZE*1.15))),
        transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Rotate +10
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=(10,10)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Rotate -10
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=(-10,-10)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Slight zoom
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.2), int(IMAGE_SIZE*1.2))),
        transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Flip + rotate
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomRotation(degrees=(15,15)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    # Brightness adjusted
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ColorJitter(brightness=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
]

class LabeledOnlyImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes, class_to_idx = super().find_classes(directory)
        classes      = [c for c in classes if c != "undefined"]
        class_to_idx = {c: i for i, c in enumerate(classes)}
        return classes, class_to_idx

class UnlabeledDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.paths[idx]

class PseudoLabeledDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir    = test_dir
        self.transform   = transform
        self.image_files = sorted([
            f for f in os.listdir(test_dir)
            if f.lower().endswith(('.jpg','.jpeg','.png'))
        ])
    def __len__(self): return len(self.image_files)
    def __getitem__(self, idx):
        fname = self.image_files[idx]
        img   = Image.open(os.path.join(self.test_dir, fname)).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, os.path.splitext(fname)[0]

print("Completed")

Completed


In [14]:
def get_model():
    m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def mixup_data(x, y, alpha=0.4):
    """Mixup augmentation — blends two images together"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index      = torch.randperm(batch_size).to(DEVICE)
    mixed_x    = lam * x + (1 - lam) * x[index, :]
    y_a, y_b   = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_model(model, train_loader, val_loader, epochs, lr, save_path, use_mixup=True):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    criterion = nn.CrossEntropyLoss()
    best_acc  = 0.0

    print(f"\n{'Epoch':>6} | {'Train Acc':>9} | {'Val Acc':>8}")
    print("-"*32)

    for epoch in range(1, epochs+1):
        model.train()
        correct, total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            if use_mixup and np.random.random() > 0.5:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels)
                out  = model(imgs)
                loss = mixup_criterion(criterion, out, y_a, y_b, lam)
            else:
                out  = model(imgs)
                loss = criterion(out, labels)

            loss.backward()
            optimizer.step()
            correct += (out.argmax(1) == labels).sum().item()
            total   += labels.size(0)
        train_acc = correct / total

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total   += labels.size(0)
        val_acc = correct / total
        scheduler.step()

        flag = " ← best" if val_acc > best_acc else ""
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
        print(f"{epoch:>6} | {train_acc:>9.4f} | {val_acc:>8.4f}{flag}")

    print(f"\nBest: {best_acc:.4f}")
    return best_acc

def pseudo_label(model, paths, threshold):
    """Label unlabeled images using model confidence"""
    model.eval()
    loader = DataLoader(
        UnlabeledDataset(paths, transform=val_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )
    sel_paths, sel_labels, skipped = [], [], 0
    with torch.no_grad():
        for imgs, fpaths in loader:
            probs        = torch.softmax(model(imgs.to(DEVICE)), dim=1)
            confs, preds = probs.max(dim=1)
            for path, pred, conf in zip(fpaths, preds, confs):
                if conf.item() >= threshold:
                    sel_paths.append(path)
                    sel_labels.append(pred.item())
                else:
                    skipped += 1
    print(f"  Threshold : {threshold} | Selected: {len(sel_paths)} | Skipped: {skipped}")
    print(f"  Chihuahua : {sel_labels.count(0)} | Muffin: {sel_labels.count(1)}")
    return sel_paths, sel_labels

print("Completed")

Completed


In [15]:
print("PHASE 1 — 100 labeled images | 50 epochs | Mixup ON \n")


train_dataset = LabeledOnlyImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_transform)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)

all_unlabeled = sorted([
    os.path.join(UNDEFINED_DIR, f)
    for f in os.listdir(UNDEFINED_DIR)
    if f.lower().endswith(('.jpg','.jpeg','.png'))
])
print(f"Unlabeled pool : {len(all_unlabeled)}")
print(f"Train          : {len(train_dataset)}")
print(f"Val            : {len(val_dataset)}")

model1     = get_model()
phase1_acc = train_model(model1, train_loader, val_loader,
                         epochs=50, lr=1e-4,
                         save_path="model1.pth", use_mixup=True)

PHASE 1 — 100 labeled images | 50 epochs | Mixup ON 

Unlabeled pool : 3633
Train          : 100
Val            : 1000

 Epoch | Train Acc |  Val Acc
--------------------------------
     1 |    0.5900 |   0.5590 ← best
     2 |    0.6400 |   0.5770 ← best
     3 |    0.6700 |   0.7000 ← best
     4 |    0.8100 |   0.7360 ← best
     5 |    0.8000 |   0.7570 ← best
     6 |    0.8000 |   0.7800 ← best
     7 |    0.7200 |   0.7920 ← best
     8 |    0.7000 |   0.7790
     9 |    0.6800 |   0.7870
    10 |    0.7100 |   0.7950 ← best
    11 |    0.8200 |   0.8070 ← best
    12 |    0.8200 |   0.8120 ← best
    13 |    0.7700 |   0.7760
    14 |    0.7100 |   0.6860
    15 |    0.7800 |   0.6420
    16 |    0.8200 |   0.6110
    17 |    0.7300 |   0.6330
    18 |    0.7800 |   0.6910
    19 |    0.7300 |   0.7540
    20 |    0.8500 |   0.7620
    21 |    0.9300 |   0.7720
    22 |    0.6600 |   0.7830
    23 |    0.5400 |   0.7820
    24 |    0.7400 |   0.7710
    25 |    0.7500 |   0.76

In [16]:
print("PHASE 2 — Pseudo label at 0.90 → maximize data \n")

model1.load_state_dict(torch.load("model1.pth"))
paths_r1, labels_r1 = pseudo_label(model1, all_unlabeled, threshold=0.90)

pseudo_r1   = PseudoLabeledDataset(paths_r1, labels_r1, transform=train_transform)
combined_r1 = ConcatDataset([train_dataset, pseudo_r1])
loader_r1   = DataLoader(combined_r1, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"\nTotal training : {len(combined_r1)}")

model2     = get_model()
phase2_acc = train_model(model2, loader_r1, val_loader,
                         epochs=50, lr=5e-5,
                         save_path="model2.pth", use_mixup=True)

PHASE 2 — Pseudo label at 0.90 → maximize data 

  Threshold : 0.9 | Selected: 1346 | Skipped: 2287
  Chihuahua : 645 | Muffin: 701

Total training : 1446

 Epoch | Train Acc |  Val Acc
--------------------------------
     1 |    0.7794 |   0.9470 ← best
     2 |    0.7898 |   0.9470
     3 |    0.8257 |   0.9310
     4 |    0.8209 |   0.9400
     5 |    0.8195 |   0.9390
     6 |    0.8679 |   0.9320
     7 |    0.8755 |   0.9020
     8 |    0.8880 |   0.9190
     9 |    0.8485 |   0.9070
    10 |    0.8008 |   0.9260
    11 |    0.8278 |   0.9310
    12 |    0.8154 |   0.9260
    13 |    0.8645 |   0.9150
    14 |    0.8479 |   0.9150
    15 |    0.8105 |   0.9000
    16 |    0.8797 |   0.9150
    17 |    0.8997 |   0.9150
    18 |    0.8769 |   0.8990
    19 |    0.8589 |   0.9140
    20 |    0.7953 |   0.9220
    21 |    0.8976 |   0.8840
    22 |    0.8568 |   0.8940
    23 |    0.8783 |   0.9020
    24 |    0.8693 |   0.9140
    25 |    0.9122 |   0.9070
    26 |    0.8050 |   0

In [17]:
print("PHASE 3 — Label remaining images at 0.85 \n")


model2.load_state_dict(torch.load("model2.pth"))

remaining_r2 = [p for p in all_unlabeled if p not in set(paths_r1)]
print(f"Remaining unlabeled : {len(remaining_r2)}")

paths_r2, labels_r2 = pseudo_label(model2, remaining_r2, threshold=0.85)

pseudo_r2   = PseudoLabeledDataset(paths_r2, labels_r2, transform=train_transform)
combined_r2 = ConcatDataset([train_dataset, pseudo_r1, pseudo_r2])
loader_r2   = DataLoader(combined_r2, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"\nTotal training : {len(combined_r2)}")
print(f"  Original     : 100")
print(f"  Round 1      : {len(pseudo_r1)}")
print(f"  Round 2      : {len(pseudo_r2)}")

model3     = get_model()
phase3_acc = train_model(model3, loader_r2, val_loader,
                         epochs=50, lr=2e-5,
                         save_path="model3.pth", use_mixup=True)

PHASE 3 — Label remaining images at 0.85 

Remaining unlabeled : 2287
  Threshold : 0.85 | Selected: 1449 | Skipped: 838
  Chihuahua : 625 | Muffin: 824

Total training : 2895
  Original     : 100
  Round 1      : 1346
  Round 2      : 1449

 Epoch | Train Acc |  Val Acc
--------------------------------
     1 |    0.7665 |   0.9560 ← best
     2 |    0.8276 |   0.9680 ← best
     3 |    0.8570 |   0.9750 ← best
     4 |    0.8083 |   0.9720
     5 |    0.8456 |   0.9750
     6 |    0.8373 |   0.9700
     7 |    0.8242 |   0.9660
     8 |    0.8408 |   0.9740
     9 |    0.8629 |   0.9760 ← best
    10 |    0.8715 |   0.9660
    11 |    0.8428 |   0.9670
    12 |    0.8525 |   0.9680
    13 |    0.8573 |   0.9710
    14 |    0.8428 |   0.9750
    15 |    0.8839 |   0.9690
    16 |    0.8442 |   0.9760
    17 |    0.8712 |   0.9770 ← best
    18 |    0.8301 |   0.9740
    19 |    0.8366 |   0.9720
    20 |    0.8587 |   0.9680
    21 |    0.8722 |   0.9700
    22 |    0.8801 |   0.9700


In [18]:
print("PHASE 4 — Label ALL remaining (threshold=0.80)")

model3.load_state_dict(torch.load("model3.pth"))

already     = set(paths_r1) | set(paths_r2)
remaining_r3 = [p for p in all_unlabeled if p not in already]
print(f"Remaining unlabeled : {len(remaining_r3)}")

paths_r3, labels_r3 = pseudo_label(model3, remaining_r3, threshold=0.80)

pseudo_r3   = PseudoLabeledDataset(paths_r3, labels_r3, transform=train_transform)
combined_r3 = ConcatDataset([train_dataset, pseudo_r1, pseudo_r2, pseudo_r3])
loader_r3   = DataLoader(combined_r3, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"\nFinal training : {len(combined_r3)}")
print(f"  Original     : 100")
print(f"  Round 1      : {len(pseudo_r1)}")
print(f"  Round 2      : {len(pseudo_r2)}")
print(f"  Round 3      : {len(pseudo_r3)}")
print(f"  Unlabeled    : {len(all_unlabeled) - len(paths_r1) - len(paths_r2) - len(paths_r3)} (too uncertain, excluded)")

model4     = get_model()
phase4_acc = train_model(model4, loader_r3, val_loader,
                         epochs=50, lr=1e-5,
                         save_path="model4.pth", use_mixup=False)  # no mixup on final

print(f"\n{'═'*45}")
print(f"Phase 1 (100 imgs)        : {phase1_acc:.4f}")
print(f"Phase 2 (+ round 1)       : {phase2_acc:.4f}")
print(f"Phase 3 (+ round 2)       : {phase3_acc:.4f}")
print(f"Phase 4 (+ round 3, all)  : {phase4_acc:.4f}")
print(f"{'═'*45}")

PHASE 4 — Label ALL remaining (threshold=0.80)
Remaining unlabeled : 838
  Threshold : 0.8 | Selected: 666 | Skipped: 172
  Chihuahua : 562 | Muffin: 104

Final training : 3561
  Original     : 100
  Round 1      : 1346
  Round 2      : 1449
  Round 3      : 666
  Unlabeled    : 172 (too uncertain, excluded)

 Epoch | Train Acc |  Val Acc
--------------------------------
     1 |    0.8110 |   0.9690 ← best
     2 |    0.9104 |   0.9810 ← best
     3 |    0.9290 |   0.9810
     4 |    0.9444 |   0.9820 ← best
     5 |    0.9472 |   0.9800
     6 |    0.9545 |   0.9800
     7 |    0.9531 |   0.9800
     8 |    0.9514 |   0.9820
     9 |    0.9548 |   0.9780
    10 |    0.9548 |   0.9810
    11 |    0.9598 |   0.9790
    12 |    0.9582 |   0.9790
    13 |    0.9579 |   0.9820
    14 |    0.9635 |   0.9810
    15 |    0.9596 |   0.9800
    16 |    0.9587 |   0.9780
    17 |    0.9584 |   0.9810
    18 |    0.9655 |   0.9810
    19 |    0.9624 |   0.9780
    20 |    0.9604 |   0.9790
    2

In [20]:
print("Submissions csv file")

model4.load_state_dict(torch.load("model4.pth"))
model4.eval()

all_probs, all_ids = [], []

for i, tta_tf in enumerate(tta_transforms):
    loader      = DataLoader(
        TestDataset(TEST_DIR, transform=tta_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )
    batch_probs, ids_temp = [], []
    with torch.no_grad():
        for imgs, ids in loader:
            probs = torch.softmax(model4(imgs.to(DEVICE)), dim=1)
            batch_probs.append(probs.cpu())
            ids_temp.extend(ids)
    all_probs.append(torch.cat(batch_probs, dim=0))
    if i == 0: all_ids = ids_temp
    print(f"TTA pass {i+1:>2}/10 ")

avg_probs   = torch.stack(all_probs).mean(dim=0)
final_preds = avg_probs.argmax(dim=1)
final_confs = avg_probs.max(dim=1).values

submission = pd.DataFrame({
    "image_id"  : all_ids,
    "prediction": final_preds.numpy(),
    "confidence": [round(float(c), 4) for c in final_confs],
})
submission.to_csv("submission.csv", index=False)

print(f"\nsubmission.csv saved  ({len(submission)} rows)")
print(submission["prediction"].value_counts())

from IPython.display import FileLink
FileLink('submission.csv')

Submissions csv file
TTA pass  1/10 
TTA pass  2/10 
TTA pass  3/10 
TTA pass  4/10 
TTA pass  5/10 
TTA pass  6/10 
TTA pass  7/10 
TTA pass  8/10 
TTA pass  9/10 
TTA pass 10/10 

submission.csv saved  (1184 rows)
prediction
0    644
1    540
Name: count, dtype: int64


/kaggle/working/submission.csv

In [22]:
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir    = test_dir
        self.transform   = transform
        self.image_files = sorted([
            f for f in os.listdir(test_dir)
            if f.lower().endswith(('.jpg','.jpeg','.png'))
        ])
    def __len__(self): return len(self.image_files)
    def __getitem__(self, idx):
        fname = self.image_files[idx]
        img   = Image.open(os.path.join(self.test_dir, fname)).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, os.path.splitext(fname)[0]

In [23]:
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

tta_transforms = [
    val_transform,
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0), transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.15), int(IMAGE_SIZE*1.15))),
        transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=(10,10)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=(-10,-10)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.2), int(IMAGE_SIZE*1.2))),
        transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomRotation(degrees=(15,15)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ColorJitter(brightness=0.3), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
]

In [24]:
def load_model(path):
    m    = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 2)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m    = m.to(DEVICE)
    m.eval()
    return m

all_models = {
    "model1": load_model("model1.pth"),
    "model2": load_model("model2.pth"),
    "model3": load_model("model3.pth"),
    "model4": load_model("model4.pth"),
}
print(f"Loaded {len(all_models)} models")

Loaded 4 models


In [25]:
all_probs = []
all_ids   = []

for model_name, model in all_models.items():
    for t_idx, tta_tf in enumerate(tta_transforms):
        loader      = DataLoader(
            TestDataset(TEST_DIR, transform=tta_tf),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=2
        )
        batch_probs = []
        ids_temp    = []

        with torch.no_grad():
            for imgs, ids in loader:
                probs = torch.softmax(model(imgs.to(DEVICE)), dim=1)
                batch_probs.append(probs.cpu())
                ids_temp.extend(ids)

        all_probs.append(torch.cat(batch_probs, dim=0))
        if len(all_ids) == 0:
            all_ids = ids_temp

        print(f"{model_name} | TTA {t_idx+1:>2}/10")

model1 | TTA  1/10
model1 | TTA  2/10
model1 | TTA  3/10
model1 | TTA  4/10
model1 | TTA  5/10
model1 | TTA  6/10
model1 | TTA  7/10
model1 | TTA  8/10
model1 | TTA  9/10
model1 | TTA 10/10
model2 | TTA  1/10
model2 | TTA  2/10
model2 | TTA  3/10
model2 | TTA  4/10
model2 | TTA  5/10
model2 | TTA  6/10
model2 | TTA  7/10
model2 | TTA  8/10
model2 | TTA  9/10
model2 | TTA 10/10
model3 | TTA  1/10
model3 | TTA  2/10
model3 | TTA  3/10
model3 | TTA  4/10
model3 | TTA  5/10
model3 | TTA  6/10
model3 | TTA  7/10
model3 | TTA  8/10
model3 | TTA  9/10
model3 | TTA 10/10
model4 | TTA  1/10
model4 | TTA  2/10
model4 | TTA  3/10
model4 | TTA  4/10
model4 | TTA  5/10
model4 | TTA  6/10
model4 | TTA  7/10
model4 | TTA  8/10
model4 | TTA  9/10
model4 | TTA 10/10


In [26]:
# avg all 40 predictions
avg_probs   = torch.stack(all_probs).mean(dim=0)
final_preds = avg_probs.argmax(dim=1)
final_confs = avg_probs.max(dim=1).values

submission = pd.DataFrame({
    "image_id"  : all_ids,
    "prediction": final_preds.numpy(),
    "confidence": [round(float(c), 4) for c in final_confs],
})
submission.to_csv("submission.csv", index=False)

print(f"\nsubmission.csv saved! ({len(submission)} rows)")
print(submission["prediction"].value_counts())
print(submission.head())

from IPython.display import FileLink
FileLink('submission.csv')


submission.csv saved! (1184 rows)
prediction
0    622
1    562
Name: count, dtype: int64
     image_id  prediction  confidence
0  test_00001           0      0.9298
1  test_00002           0      0.8462
2  test_00003           0      0.9334
3  test_00004           0      0.9567
4  test_00005           0      0.5822


/kaggle/working/submission.csv

In [27]:
import os, random, torch, torch.nn as nn, torch.optim as optim
import pandas as pd, numpy as np
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision import datasets, transforms, models
from PIL import Image

In [28]:
DATA_DIR      = "/kaggle/input/competitions/3-lc-x-mnnit-ai-hackathon/Starter_Kit/data"
TRAIN_DIR     = DATA_DIR + "/train"
VAL_DIR       = DATA_DIR + "/val"
TEST_DIR      = DATA_DIR + "/test"
UNDEFINED_DIR = TRAIN_DIR + "/undefined"
IMAGE_SIZE, BATCH_SIZE = 128, 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"Device: {DEVICE}")

Device: cpu


In [29]:
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.15),
    transforms.RandomGrayscale(p=0.15),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.2),
])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tta_tfs = [
    val_tf,
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomHorizontalFlip(p=1.0),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomVerticalFlip(p=1.0),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomHorizontalFlip(p=1.0),transforms.RandomVerticalFlip(p=1.0),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.15),int(IMAGE_SIZE*1.15))),transforms.CenterCrop(IMAGE_SIZE),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomRotation(degrees=(10,10)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomRotation(degrees=(-10,-10)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.2),int(IMAGE_SIZE*1.2))),transforms.CenterCrop(IMAGE_SIZE),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomHorizontalFlip(p=1.0),transforms.RandomRotation(degrees=(15,15)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ColorJitter(brightness=0.3),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]

In [30]:
# dataset classes
class LabeledFolder(datasets.ImageFolder):
    def find_classes(self, d):
        c, ci = super().find_classes(d)
        c  = [x for x in c if x != "undefined"]
        ci = {x:i for i,x in enumerate(c)}
        return c, ci

class UnlabeledDS(Dataset):
    def __init__(self, paths, tf=None):
        self.paths=paths; self.tf=tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return (self.tf(img) if self.tf else img), self.paths[i]

class PseudoDS(Dataset):
    def __init__(self, paths, labels, tf=None):
        self.paths=paths; self.labels=labels; self.tf=tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return (self.tf(img) if self.tf else img), self.labels[i]

class TestDS(Dataset):
    def __init__(self, d, tf=None):
        self.d=d; self.tf=tf
        self.files=sorted([f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.d,self.files[i])).convert("RGB")
        return (self.tf(img) if self.tf else img), os.path.splitext(self.files[i])[0]

# functs
def get_model(seed=42):
    set_seed(seed)
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def train_model(model, tr_loader, va_loader, epochs, lr, path, mixup=True):
    opt  = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-7)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    best = 0.0
    print(f"\n{'Ep':>4} | {'TrAcc':>6} | {'VaAcc':>6}")
    print("-"*24)
    for ep in range(1, epochs+1):
        model.train()
        cor=tot=0
        for imgs,labels in tr_loader:
            imgs,labels = imgs.to(DEVICE),labels.to(DEVICE)
            opt.zero_grad()
            if mixup and np.random.random()>0.5:
                lam=np.random.beta(0.4,0.4); idx=torch.randperm(imgs.size(0)).to(DEVICE)
                imgs2=lam*imgs+(1-lam)*imgs[idx]
                out=model(imgs2)
                loss=lam*crit(out,labels)+(1-lam)*crit(out,labels[idx])
            else:
                out=model(imgs); loss=crit(out,labels)
            loss.backward(); opt.step()
            cor+=(out.argmax(1)==labels).sum().item(); tot+=labels.size(0)
        model.eval(); vc=vt=0
        with torch.no_grad():
            for imgs,labels in va_loader:
                imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
                vc+=(model(imgs).argmax(1)==labels).sum().item(); vt+=labels.size(0)
        vacc=vc/vt; sch.step()
        if vacc>best:
            best=vacc; torch.save(model.state_dict(),path)
        print(f"{ep:>4} | {cor/tot:>6.4f} | {vacc:>6.4f}{' ←' if vacc==best else ''}")
    print(f"Best: {best:.4f}")
    return best

def pseudo_label(mdls, paths, thr):
    for m in mdls: m.eval()
    loader = DataLoader(UnlabeledDS(paths,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
    sp,sl,sk = [],[],0
    with torch.no_grad():
        for imgs,fps in loader:
            imgs=imgs.to(DEVICE)
            probs=torch.stack([torch.softmax(m(imgs),dim=1) for m in mdls]).mean(0)
            confs,preds=probs.max(1)
            for p,pred,conf in zip(fps,preds,confs):
                if conf.item()>=thr: sp.append(p); sl.append(pred.item())
                else: sk+=1
    print(f"  thr={thr} | selected={len(sp)} (chi={sl.count(0)},muf={sl.count(1)}) | skipped={sk}")
    return sp,sl

In [31]:
train_ds = LabeledFolder(TRAIN_DIR, transform=train_tf)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=val_tf)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
all_unlabeled = sorted([os.path.join(UNDEFINED_DIR,f)
    for f in os.listdir(UNDEFINED_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f"Labeled={len(train_ds)} | Val={len(val_ds)} | Unlabeled={len(all_unlabeled)}")

Labeled=100 | Val=1000 | Unlabeled=3633


In [32]:
print("PHASE 1 — 100 labeled images × 3 seeds\n")
tr_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
p1_models, p1_accs = [], []
for seed in [42, 123, 777]:
    print(f"\n -- seed={seed}")
    m=get_model(seed); acc=train_model(m,tr_loader,val_loader,epochs=40,lr=1e-4,path=f"p1_{seed}.pth")
    m.load_state_dict(torch.load(f"p1_{seed}.pth")); p1_models.append(m); p1_accs.append(acc)
print(f"\nPhase 1 accs: {[f'{a:.4f}' for a in p1_accs]}")


PHASE 1 — 100 labeled images × 3 seeds


── seed=42

  Ep |  TrAcc |  VaAcc
------------------------
   1 | 0.4600 | 0.7730 ←
   2 | 0.7400 | 0.7870 ←
   3 | 0.6700 | 0.7920 ←
   4 | 0.6700 | 0.8050 ←
   5 | 0.7300 | 0.7750
   6 | 0.8300 | 0.7420
   7 | 0.8400 | 0.7360
   8 | 0.8700 | 0.7540
   9 | 0.6500 | 0.7690
  10 | 0.5900 | 0.7540
  11 | 0.8600 | 0.7340
  12 | 0.7300 | 0.7120
  13 | 0.8000 | 0.7150
  14 | 0.7200 | 0.6980
  15 | 0.8400 | 0.6820
  16 | 0.8600 | 0.6890
  17 | 0.8100 | 0.7030
  18 | 0.8400 | 0.7240
  19 | 0.8600 | 0.7550
  20 | 0.5700 | 0.7790
  21 | 0.9100 | 0.7750
  22 | 0.6400 | 0.7570
  23 | 0.9200 | 0.7750
  24 | 0.7900 | 0.7790
  25 | 0.9000 | 0.7830
  26 | 0.7600 | 0.7890
  27 | 0.9400 | 0.7920
  28 | 0.7700 | 0.7920
  29 | 0.7600 | 0.7800
  30 | 0.8300 | 0.7930
  31 | 0.9400 | 0.7980
  32 | 0.7000 | 0.7920
  33 | 0.9300 | 0.8060 ←
  34 | 0.6600 | 0.8100 ←
  35 | 0.8900 | 0.8150 ←
  36 | 0.8700 | 0.8100
  37 | 0.8600 | 0.8050
  38 | 0.9500 | 0.8080
  39 | 0.93

In [33]:
print("PHASE 2 — pseudo label thr=0.95\n")
paths_r1,labels_r1 = pseudo_label(p1_models, all_unlabeled, 0.95)
pseudo_r1   = PseudoDS(paths_r1,labels_r1,train_tf)
combined_r1 = ConcatDataset([train_ds,pseudo_r1])
loader_r1   = DataLoader(combined_r1,batch_size=BATCH_SIZE,shuffle=True,num_workers=2)
print(f"Total training: {len(combined_r1)}")
p2_models, p2_accs = [], []
for seed in [42, 123, 777]:
    print(f"\n -- seed={seed}")
    m=get_model(seed); acc=train_model(m,loader_r1,val_loader,epochs=40,lr=5e-5,path=f"p2_{seed}.pth")
    m.load_state_dict(torch.load(f"p2_{seed}.pth")); p2_models.append(m); p2_accs.append(acc)
print(f"\nPhase 2 accs: {[f'{a:.4f}' for a in p2_accs]}")

PHASE 2 — pseudo label thr=0.95

  thr=0.95 | selected=327 (chi=127,muf=200) | skipped=3306
Total training: 427

 -- seed=42

  Ep |  TrAcc |  VaAcc
------------------------
   1 | 0.7681 | 0.9380 ←
   2 | 0.8407 | 0.9300
   3 | 0.8080 | 0.9130
   4 | 0.8525 | 0.8660
   5 | 0.8501 | 0.8920
   6 | 0.8665 | 0.9050
   7 | 0.8689 | 0.9000
   8 | 0.8126 | 0.8740
   9 | 0.8126 | 0.8880
  10 | 0.8290 | 0.8970
  11 | 0.9204 | 0.9130
  12 | 0.8548 | 0.8960
  13 | 0.7588 | 0.8800
  14 | 0.8806 | 0.8930
  15 | 0.8689 | 0.8740
  16 | 0.8618 | 0.8680
  17 | 0.9391 | 0.8840
  18 | 0.8571 | 0.8740
  19 | 0.9040 | 0.8650
  20 | 0.9719 | 0.8730
  21 | 0.8173 | 0.8660
  22 | 0.9532 | 0.8710
  23 | 0.8126 | 0.8650
  24 | 0.8712 | 0.8560
  25 | 0.9040 | 0.8640
  26 | 0.8642 | 0.8630
  27 | 0.9297 | 0.8630
  28 | 0.8009 | 0.8680
  29 | 0.8548 | 0.8610
  30 | 0.8314 | 0.8720
  31 | 0.8595 | 0.8640
  32 | 0.8431 | 0.8610
  33 | 0.8407 | 0.8610
  34 | 0.8642 | 0.8680
  35 | 0.9438 | 0.8710
  36 | 0.8478 | 0.8

In [ ]:
print("PHASE 3 — pseudo label thr=0.85 on remaining\n")
remaining_r2 = [p for p in all_unlabeled if p not in set(paths_r1)]
paths_r2,labels_r2 = pseudo_label(p2_models, remaining_r2, 0.85)
pseudo_r2   = PseudoDS(paths_r2,labels_r2,train_tf)
combined_r2 = ConcatDataset([train_ds,pseudo_r1,pseudo_r2])
loader_r2   = DataLoader(combined_r2,batch_size=BATCH_SIZE,shuffle=True,num_workers=2)
print(f"Total training: {len(combined_r2)} (orig=100, r1={len(pseudo_r1)}, r2={len(pseudo_r2)})")
p3_models, p3_accs = [], []
for seed in [42, 123, 777]:
    print(f"\n -- seed={seed}")
    m=get_model(seed); acc=train_model(m,loader_r2,val_loader,epochs=20,lr=2e-5,path=f"p3_{seed}.pth")
    m.load_state_dict(torch.load(f"p3_{seed}.pth")); p3_models.append(m); p3_accs.append(acc)
print(f"\nPhase 3 accs: {[f'{a:.4f}' for a in p3_accs]}")
print(f"\n{'═'*45}")
print(f"Phase 1 best : {max(p1_accs):.4f}")
print(f"Phase 2 best : {max(p2_accs):.4f}")
print(f"Phase 3 best : {max(p3_accs):.4f}")
print(f"{'═'*45}")

PHASE 3 — pseudo label thr=0.85 on remaining

  thr=0.85 | selected=1709 (chi=907,muf=802) | skipped=1597
Total training: 2136 (orig=100, r1=327, r2=1709)

 -- seed=42

  Ep |  TrAcc |  VaAcc
------------------------
   1 | 0.7814 | 0.9590 ←
   2 | 0.8390 | 0.9670 ←
   3 | 0.8394 | 0.9630
   4 | 0.8727 | 0.9620
   5 | 0.8694 | 0.9670 ←
   6 | 0.8647 | 0.9710 ←
   7 | 0.8558 | 0.9700
   8 | 0.8436 | 0.9700
   9 | 0.8624 | 0.9680
  10 | 0.8488 | 0.9750 ←
  11 | 0.8591 | 0.9790 ←
  12 | 0.8993 | 0.9750
  13 | 0.8596 | 0.9760
  14 | 0.8408 | 0.9790 ←
  15 | 0.8820 | 0.9800 ←
  16 | 0.8820 | 0.9770
  17 | 0.8783 | 0.9770
  18 | 0.8301 | 0.9780
  19 | 0.8699 | 0.9780
  20 | 0.8750 | 0.9780
Best: 0.9800

 -- seed=123

  Ep |  TrAcc |  VaAcc
------------------------
   1 | 0.7856 | 0.9420 ←
   2 | 0.7992 | 0.9590 ←
   3 | 0.8713 | 0.9600 ←
   4 | 0.8535 | 0.9600 ←
   5 | 0.8450 | 0.9660 ←
   6 | 0.8422 | 0.9710 ←
   7 | 0.8811 | 0.9730 ←
   8 | 0.8886 | 0.9700
   9 | 0.8521 | 0.9670
  10 | 0.8

In [36]:
print("\nFINAL SUBMISSION — 3 models × 5 TTA\n")
all_probs, all_ids = [], []
for mi, model in enumerate(p3_models):
    model.eval()
    for ti, tta_tf in enumerate(tta_tfs):
        loader = DataLoader(TestDS(TEST_DIR,tta_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
        bp, it = [], []
        with torch.no_grad():
            for imgs,ids in loader:
                bp.append(torch.softmax(model(imgs.to(DEVICE)),dim=1).cpu()); it.extend(ids)
        all_probs.append(torch.cat(bp,dim=0))
        if mi==0 and ti==0: all_ids=it
        print(f"Model {mi+1}/3 | TTA {ti+1:>2}/5")

avg   = torch.stack(all_probs).mean(0)
preds = avg.argmax(1); confs = avg.max(1).values

submission = pd.DataFrame({
    "image_id"  : all_ids,
    "prediction": preds.numpy(),
    "confidence": [round(float(c),4) for c in confs],
})
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv saved! ({len(submission)} rows)")
print(submission["prediction"].value_counts())


FINAL SUBMISSION — 3 models × 10 TTA

Model 1/3 | TTA  1/5
Model 1/3 | TTA  2/5
Model 1/3 | TTA  3/5
Model 1/3 | TTA  4/5
Model 1/3 | TTA  5/5
Model 1/3 | TTA  6/5
Model 1/3 | TTA  7/5
Model 1/3 | TTA  8/5
Model 1/3 | TTA  9/5
Model 1/3 | TTA 10/5

submission.csv saved! (1184 rows)
prediction
0    636
1    548
Name: count, dtype: int64


In [37]:
from IPython.display import FileLink
FileLink('submission.csv')

/kaggle/working/submission.csv

In [39]:
import os
for f in os.listdir('.'):
    if f.endswith('.csv'):
        print(f)

submission.csv
